# Camada Gold — Dimensões (Star Schema)

Este notebook constrói e mantém as tabelas de **dimensão** da camada Gold, a partir da tabela enriquecida `{catalog}.gold.tb_sales_enriched` (fonte consolidada de vendas).

## O que ele faz

1. **Parametrização**: recebe o catálogo de destino via widget `CATALOG` e define o schema `gold`.
2. **Função utilitária**: define `merge()`, que aplica `MERGE INTO` (upsert — update de existentes + insert de novos) numa tabela destino a partir de uma consulta SQL e de uma chave.
3. **Criação das dimensões** (`CREATE TABLE IF NOT EXISTS`):
   - `dim_date` — datas de venda, com ano, trimestre, mês, ano-mês e `week_of_month` (período/semana do mês, 1 a 5);
   - `dim_brand` — marcas (`ce_brand_flvr` / `brand_nm`);
   - `dim_region` — regiões (`btlr_org_lvl_c_desc`);
   - `dim_channel` — canais (`trade_chnl_desc`, `chnl_group` para vendas; `trade_group_desc`/`trade_type_desc` do arquivo de canal);
   - `dim_package` — embalagens (`tsr_pckg_nm`, `pkg_cat`).
4. **Carga incremental**: cada dimensão é populada via `merge()`, com deduplicação por chave surrogate (`xxhash64` dos atributos naturais, ou `date_key` derivado da data) mantendo o registro mais recente por `_gold_updated_at`, e tratamento de nulos via `COALESCE` para `'UNKNOWN'`.

## Como usar

1. Defina o widget `CATALOG` com o catálogo de destino.
2. Execute as células em ordem: parametrização → funções → criação das tabelas → merges de carga.

> **Observação**: as chaves surrogate das dimensões (exceto `dim_date`) são hashes dos atributos naturais, garantindo consistência entre cargas.


In [0]:
dbutils.widgets.text("CATALOG", "")

CATALOG = dbutils.widgets.get("CATALOG")
G = f"{CATALOG}.gold"
SOURCE = f"{G}.tb_sales_enriched"



In [0]:
def merge(target: str, select_sql: str, key: str):
    """MERGE INTO (update + insert) da consulta `select_sql` na tabela `target`, pela chave `key`."""
    spark.sql(f"""
        MERGE INTO {target} AS t
        USING ({select_sql}) AS s
        ON t.{key} = s.{key}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {G}.dim_date (
    date_key      INT,
    sale_date     DATE,
    year          INT,
    quarter       INT,
    month         INT,
    month_name    STRING,
    year_month    STRING,
    week_of_month INT
) USING DELTA COMMENT 'Dimensão de datas. week_of_month = period (semana do mês, 1 a 5).'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {G}.dim_brand (
    brand_key     BIGINT,
    ce_brand_flvr INT,
    brand_nm      STRING
) USING DELTA COMMENT 'Dimensão de marcas.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {G}.dim_region (
    region_key  BIGINT,
    region_name STRING
) USING DELTA COMMENT 'Dimensão de regiões (btlr_org_lvl_c_desc).'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {G}.dim_channel (
    channel_key      BIGINT,
    trade_chnl_desc  STRING,
    chnl_group       STRING,
    trade_group_desc STRING,
    trade_type_desc  STRING
) USING DELTA COMMENT 'Dimensão de canais: chnl_group (vendas) e trade_group_desc/trade_type_desc (arquivo de canal).'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {G}.dim_package (
    package_key BIGINT,
    tsr_pckg_nm STRING,
    pkg_cat     STRING,
    pkg_cat_desc STRING
) USING DELTA COMMENT 'Dimensão de embalagens.'
""")

In [0]:
merge(f"{G}.dim_date", f"""
    SELECT CAST(date_format(sale_date, 'yyyyMMdd') AS INT) AS date_key,
           sale_date,
           year(sale_date)                    AS year,
           quarter(sale_date)                 AS quarter,
           month(sale_date)                   AS month,
           date_format(sale_date, 'MMMM')     AS month_name,
           date_format(sale_date, 'yyyy-MM')  AS year_month,
           period                             AS week_of_month
    FROM (SELECT sale_date, period,
                 ROW_NUMBER() OVER (PARTITION BY sale_date ORDER BY _gold_updated_at DESC) AS rn
          FROM {SOURCE} WHERE sale_date IS NOT NULL)
    WHERE rn = 1
""", "date_key")


In [0]:
merge(f"{G}.dim_brand", f"""
    SELECT xxhash64(COALESCE(CAST(ce_brand_flvr AS STRING), 'UNKNOWN')) AS brand_key,
           ce_brand_flvr,
           COALESCE(brand_nm, 'UNKNOWN') AS brand_nm
    FROM (SELECT ce_brand_flvr, brand_nm,
                 ROW_NUMBER() OVER (PARTITION BY COALESCE(CAST(ce_brand_flvr AS STRING), 'UNKNOWN')
                                    ORDER BY _gold_updated_at DESC) AS rn
          FROM {SOURCE})
    WHERE rn = 1
""", "brand_key")


In [0]:
merge(f"{G}.dim_region", f"""
    SELECT DISTINCT
           xxhash64(COALESCE(btlr_org_lvl_c_desc, 'UNKNOWN')) AS region_key,
           COALESCE(btlr_org_lvl_c_desc, 'UNKNOWN')           AS region_name
    FROM {SOURCE}
""", "region_key")

In [0]:
merge(f"{G}.dim_channel", f"""
    SELECT xxhash64(COALESCE(trade_chnl_desc, 'UNKNOWN')) AS channel_key,
           COALESCE(trade_chnl_desc, 'UNKNOWN')  AS trade_chnl_desc,
           COALESCE(chnl_group, 'UNKNOWN')       AS chnl_group,
           COALESCE(trade_group_desc, 'UNKNOWN') AS trade_group_desc,
           COALESCE(trade_type_desc, 'UNKNOWN')  AS trade_type_desc
    FROM (SELECT trade_chnl_desc, chnl_group, trade_group_desc, trade_type_desc,
                 ROW_NUMBER() OVER (PARTITION BY COALESCE(trade_chnl_desc, 'UNKNOWN')
                                    ORDER BY _gold_updated_at DESC) AS rn
          FROM {SOURCE})
    WHERE rn = 1
""", "channel_key")

In [0]:
merge(f"{G}.dim_package", f"""
    SELECT xxhash64(COALESCE(tsr_pckg_nm, 'UNKNOWN')) AS package_key,
           COALESCE(tsr_pckg_nm, 'UNKNOWN') AS tsr_pckg_nm,
           COALESCE(pkg_cat, 'UNKNOWN')     AS pkg_cat,
           COALESCE(pkg_cat_desc, 'UNKNOWN') AS pkg_cat_desc
    FROM (SELECT tsr_pckg_nm, pkg_cat, pkg_cat_desc,
                 ROW_NUMBER() OVER (PARTITION BY COALESCE(tsr_pckg_nm, 'UNKNOWN')
                                    ORDER BY _gold_updated_at DESC) AS rn
          FROM {SOURCE})
    WHERE rn = 1
""", "package_key")